# dARK Core Minter API Test

This notebook demonstrates the lifecycle of ARK identifiers using the Minter API endpoints.
It includes authority registration via the Admin API and examples of both **JSON** and **XML** metadata formats.

In [ ]:
import requests
import json
import uuid
import time

ADMIN_URL = "http://localhost:8000/api/v1/admin"
MINTER_URL = "http://localhost:8001/api/v1"

# Unique IDs for this run
AUTHORITY_ID = f"test-authority-{int(time.time())}"
NAAN = "12345"
print(f"Authority ID: {AUTHORITY_ID}")

## 0. Setup Authority
Register the authority and authorize the NAAN using the Admin API.

In [ ]:
print(f"Registering authority: {AUTHORITY_ID}...")
reg_payload = {
    "uuid": AUTHORITY_ID,
    "naans": [NAAN],
    "fund_amount_eth": 0.05
}

try:
    response = requests.post(f"{ADMIN_URL}/authority", json=reg_payload)
    print(f"Status: {response.status_code}")
    print(json.dumps(response.json(), indent=2))
except Exception as e:
    print(f"Error: {e}. Is the Admin API running on port 8000?")

## 1. Reserve Single ARK

In [ ]:
payload = {
    "authority_id": AUTHORITY_ID,
    "naan": NAAN,
    "alternate_identifiers": [
        {"schema": "doi", "value": f"10.1000/test-{uuid.uuid4().hex[:8]}"}
    ]
}

response = requests.post(f"{MINTER_URL}/arks", json=payload)
print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))
ark_id = response.json().get("ark")
print(f"\nReserved ARK: {ark_id}")

---
## 2. Update Metadata with JSON Format

Update the ARK with **JSON** metadata. This transitions the ARK to DRAFT state.

The API now requires:
- `metadata`: Raw content string (JSON or XML)
- `metadata_format`: Either `"json"` or `"xml"`

In [ ]:
# JSON metadata as a string
json_metadata = json.dumps({
    "title": "Notebook Test Record (JSON)",
    "creator": "Jupyter Notebook",
    "date": "2026-02-07",
    "description": "Example metadata in JSON format"
})

print("JSON Metadata:")
print(json_metadata)
print()

In [ ]:
update_payload = {
    "authority_id": AUTHORITY_ID,
    "target": "https://example.com/notebook-test-json",
    "metadata": json_metadata,
    "metadata_format": "json",  # <-- Specify format
    "alternate_identifiers": [
        {"schema": "doi", "value": "10.1000/test-notebook"},
        {"schema": "internal", "value": "nb-001"}
    ]
}

response = requests.put(f"{MINTER_URL}/arks/{ark_id}", json=update_payload)
print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))
print(f"\n✅ Metadata CID: {response.json().get('metadata_cid')}")
print(f"✅ Metadata Format: {response.json().get('metadata_format')}")

## 3. Resolve ARK

In [ ]:
response = requests.get(f"{MINTER_URL}/arks/{ark_id}")
print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))

---
## 4. Update Metadata with XML Format (Dublin Core)

Reserve a new ARK and update with **XML** metadata using Dublin Core (OAI-DC) format.

In [ ]:
# Reserve a new ARK for XML example
payload = {
    "authority_id": AUTHORITY_ID,
    "naan": NAAN
}
response = requests.post(f"{MINTER_URL}/arks", json=payload)
xml_ark_id = response.json().get("ark")
print(f"Reserved ARK for XML: {xml_ark_id}")

In [ ]:
# XML metadata (Dublin Core OAI-DC format)
xml_metadata = '''<?xml version="1.0" encoding="UTF-8"?>
<oai_dc:dc xmlns:oai_dc="http://www.openarchives.org/OAI/2.0/oai_dc/"
           xmlns:dc="http://purl.org/dc/elements/1.1/">
    <dc:title>XML Metadata Example (Dublin Core)</dc:title>
    <dc:creator>dARK Notebook</dc:creator>
    <dc:subject>Persistent Identifiers</dc:subject>
    <dc:description>Example metadata in XML format using Dublin Core</dc:description>
    <dc:date>2026-02-07</dc:date>
    <dc:type>Dataset</dc:type>
    <dc:format>application/xml</dc:format>
    <dc:identifier>ark:/12345/example</dc:identifier>
</oai_dc:dc>'''

print("XML Metadata (Dublin Core):")
print(xml_metadata)

In [ ]:
update_payload = {
    "authority_id": AUTHORITY_ID,
    "target": "https://example.com/xml-metadata-test",
    "metadata": xml_metadata,
    "metadata_format": "xml"  # <-- Specify XML format
}

response = requests.put(f"{MINTER_URL}/arks/{xml_ark_id}", json=update_payload)
print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))
print(f"\n✅ Metadata CID: {response.json().get('metadata_cid')}")
print(f"✅ Metadata Format: {response.json().get('metadata_format')}")

---
## 5. Batch Reserve

In [ ]:
batch_payload = {
    "authority_id": AUTHORITY_ID,
    "naan": NAAN,
    "items": [
        {"target": "https://example.com/b1", "alternate_identifiers": [{"schema": "idx", "value": "1"}], "client_item_id": "req-1"},
        {"target": "https://example.com/b2", "client_item_id": "req-2"}
    ]
}

response = requests.post(f"{MINTER_URL}/arks/batch", json=batch_payload)
print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))

---
## 6. Tombstone (Deactivate ARK)

In [ ]:
response = requests.delete(f"{MINTER_URL}/arks/{ark_id}")
print(f"Status: {response.status_code}")
print(f"ARK {ark_id} tombstoned")